# AAI551 SecondOrder Scenario
Elaine Liu. Illustrative policies and deterministic model outputs, developed with AI assistance. Read docs/model_spec.md for units and limitations. Run all cells from a fresh Python 3.12 kernel.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from dataclasses import replace
from scenario import EconomicScenario, ModelAssumptions
from simulator import ScenarioSimulator
from io_utils import load_scenarios, export_run
from analysis import compare_runs, sensitivity_sweep, crossing_brackets
from visualization import plot_runs
policies = load_scenarios("data/scenarios.json")
runs = {p.name: ScenarioSimulator(p).run() for p in policies}
print(*policies, sep="\n")
print("Fieldwise equality:", policies[0] == replace(policies[0]))

no_policy: UBI $0/month, AI tax 0%
ubi_500: UBI $500/month, AI tax 0%
tax_20: UBI $0/month, AI tax 20%
combined: UBI $500/month, AI tax 20%
Fieldwise equality: True


In [2]:
runs["no_policy"].round(4)

    year  adoption  ...  concentration_clipped  out_of_domain
0      0    0.0000  ...                  False          False
1      1    0.1000  ...                  False          False
2      2    0.1900  ...                  False          False
3      3    0.2710  ...                  False          False
4      4    0.3439  ...                  False          False
5      5    0.4095  ...                  False          False
6      6    0.4686  ...                  False          False
7      7    0.5217  ...                  False          False
8      8    0.5695  ...                  False          False
9      9    0.6126  ...                  False          False
10    10    0.6513  ...                  False          False

[11 rows x 21 columns]


In [3]:
fig = plot_runs(runs)
Path("outputs").mkdir(exist_ok=True)
fig.savefig("outputs/policy_comparison.png", dpi=140)
fig

Figure(1100x400)


## Paired comparison and demand feedback
Differences are candidate minus reference. Monetary values are annual thousand USD per model household; labor utilization differences are percentage points.

In [4]:
comparison = compare_runs(runs["no_policy"], runs["ubi_500"])
print(comparison.round(4).to_string(index=False))
no_feedback = ScenarioSimulator(policies[1], ModelAssumptions(feedback=False)).run()
print("UBI final employment, feedback on/off:", runs["ubi_500"].employment.iloc[-1], no_feedback.employment.iloc[-1])

 year  employment  income  balance  consumption
    0      0.0000  0.0000   0.0000       0.0000
    1      0.0000  6.0000  -6.0000       4.5000
    2      3.6850  7.9523  -5.2063       5.9642
    3      4.7190  8.5243  -4.9823       6.3932
    4      4.9031  8.6470  -4.9414       6.4853
    5      4.8349  8.6332  -4.9550       6.4749
    6      4.7048  8.5837  -4.9822       6.4378
    7      4.5703  8.5293  -5.0105       6.3970
    8      4.4465  8.4785  -5.0365       6.3589
    9      4.3364  8.4331  -5.0597       6.3248
   10      4.2393  8.3929  -5.0803       6.2947
UBI final employment, feedback on/off: 69.75979515113174 71.8134599207086


In [5]:
sweep = sensitivity_sweep(policies[0], "ubi", np.linspace(0, 1000, 11))
print(sweep.round(4).to_string(index=False))
substitutions = np.linspace(0, 60, 61)
fixed_target = runs["no_policy"].employment.iloc[-1]
differences = [ScenarioSimulator(replace(policies[1], substitution=float(x))).run().employment.iloc[-1] - fixed_target for x in substitutions]
print("Substitution brackets against fixed reference employment:", crossing_brackets(substitutions, differences))

   ubi  mean_employment  mean_income
   0.0          76.1521      51.6646
 100.0          76.9606      53.3074
 200.0          77.7692      54.9506
 300.0          78.5780      56.5941
 400.0          79.3869      58.2379
 500.0          80.1960      59.8821
 600.0          81.0052      61.5266
 700.0          81.8146      63.1715
 800.0          82.6241      64.8167
 900.0          83.4337      66.4622
1000.0          84.2435      68.1081
Substitution brackets against fixed reference employment: [(30.0, 31.0)]


## Exception handling and meaningful export
Invalid inputs and absent files have different exception types. The export sidecar preserves the configuration.

In [6]:
try:
    EconomicScenario(tax=41)
except ValueError as exc:
    print("Invalid parameter handled:", exc)
try:
    load_scenarios("data/intentional_missing_file.json")
except FileNotFoundError as exc:
    print("Missing input handled:", exc.filename)
sim = ScenarioSimulator(policies[1])
export_run(runs["ubi_500"], sim, "outputs/ubi_500.csv")
restored = pd.read_csv("outputs/ubi_500.csv")
np.testing.assert_allclose(restored.income, runs["ubi_500"].income)
print("CSV round trip verified:", len(restored), "annual rows")

Invalid parameter handled: tax must be finite and in [0, 40]
Missing input handled: data/intentional_missing_file.json
CSV round trip verified: 11 annual rows


## Research status
A separate 100-run exploratory pilot is saved under outputs/pilot. Run `python -m research.run_pilot` to rebuild. Main E1–E5 experiments, reviewed proofs, literature novelty review and the complete paper remain pending. Results describe this model and its selected assumptions.